# 🦀 CRust — C-to-Rust Migration RL Training

**Meta PyTorch OpenEnv Hackathon 2026 | Theme #2: Super Long-Horizon Planning & Instruction Following**

Trains **Qwen 2.5 3B** via **GRPO** (Group Relative Policy Optimization) to migrate legacy C code to memory-safe, modular Rust.

| Component | Where it runs |
|---|---|
| **OpenEnv Server** (for judges) | HF Space A10G — `https://adithyakommuri-meta-hackathon-final.hf.space` |
| **GRPO Training** | This Colab (T4 GPU, free) |
| **Reward function** | Real `cargo check` + `cargo test` — no LLM judge |

**⏱ Expected time**: ~60 min for 100 steps on T4 · ~30 min on A10G

> **Runtime → Change runtime type → GPU (T4)**

## Cell 1 — Install dependencies (~5 min)

In [ ]:
import os, subprocess, sys

# ── 1a. Python packages ──────────────────────────────────────────────────────
print('Installing Python packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    "openenv-core",
    'transformers>=4.40.0,<4.60.0',
    'trl>=0.9.0,<0.20.0',
    'peft>=0.10.0',
    'accelerate>=0.27.0',
    'datasets>=2.18.0',
    'huggingface_hub',
    'requests',
    'matplotlib',
    'bitsandbytes',   # 4-bit quantisation on T4
], check=True)

# ── 1b. Rust toolchain (needed for cargo check / cargo test) ────────────────
print('Installing Rust toolchain...')
subprocess.run(
    'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --no-modify-path',
    shell=True, check=True
)
cargo_bin = os.path.expanduser('~/.cargo/bin')
os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')

result = subprocess.run(['cargo', '--version'], capture_output=True, text=True)
print('Rust ready:', result.stdout.strip())
print('\n✅ All dependencies installed!')

## Cell 2 — Clone CRust project from GitHub

In [ ]:
import os, sys

GITHUB_REPO = 'https://github.com/22adi66/meta_pytorch_scalar_hackathon.git'
PROJECT_DIR = '/content/crust_project'

if not os.path.exists(PROJECT_DIR):
    os.system(f'git clone {GITHUB_REPO} {PROJECT_DIR}')
else:
    os.system(f'cd {PROJECT_DIR} && git pull')

sys.path.insert(0, PROJECT_DIR)

WORKSPACE = f'{PROJECT_DIR}/crust_env/dummy_workspace'
os.environ['CRUST_WORKSPACE'] = WORKSPACE
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ.get('PATH', '')

print('Project path:', PROJECT_DIR)
print('Workspace:   ', WORKSPACE)
print('Files:', os.listdir(f'{PROJECT_DIR}/crust_env'))

## Cell 3 — Verify the CRust environment works locally

In [ ]:
import json
from crust_env.env import MigrationEnv

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])

obs = env.reset(phase=1)
print('=== Environment Reset OK ===')
print('Target file  :', obs['current_target'])
print('Constraints  :', obs['constraints'])
print('Phase        :', obs['phase'])
print('Files remain :', obs['files_remaining'])
print()
print('C source preview:')
print(obs['c_source_code'][:400])

In [ ]:
# Verify reward function works end-to-end
obs = env.reset(phase=1)
bad = env.step({'file_path': 'src/math_ops.rs', 'code_content': 'fn main() {}'})
print(f'Bad code reward : {bad["reward"]}   stage={bad["info"]["verification"]["stage"]}')

obs = env.reset(phase=1)
good_rust = '''
pub fn add(a: i32, b: i32) -> i32 { a + b }
pub fn subtract(a: i32, b: i32) -> i32 { a - b }
pub fn multiply(a: i32, b: i32) -> i32 { a * b }
pub fn divide(a: i32, b: i32) -> Option<i32> {
    if b == 0 { None } else { Some(a / b) }
}
pub fn clamp(value: i32, min_val: i32, max_val: i32) -> i32 {
    value.max(min_val).min(max_val)
}
'''
good = env.step({'file_path': 'src/math_ops.rs', 'code_content': good_rust})
print(f'Good code reward: {good["reward"]}   stage={good["info"]["verification"]["stage"]}')
print(f'Breakdown: {json.dumps(good["info"]["reward_breakdown"], indent=2)}')
print('\n✅ Environment + reward function verified!')

## Cell 4 — Load model (Qwen 2.5 3B, 4-bit on T4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME     = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048

IS_A10G = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 20e9
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Using {"bf16 full precision" if IS_A10G else "4-bit QLoRA"}')

if IS_A10G:
    # A10G (24 GB) — load in bf16, no quantisation needed
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )
else:
    # T4 (16 GB) — 4-bit NF4 quantisation
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_cfg,
        device_map='auto',
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Apply LoRA adapters
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

if torch.cuda.is_available():
    vram  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used: {vram:.1f} GB / {total:.1f} GB')
print('✅ Model ready!')

## Cell 5 — Build dataset and reward function

In [ ]:
import re
from datasets import Dataset
from crust_env.env import MigrationEnv

SYSTEM_PROMPT = """You are an expert Rust systems programmer specializing in memory-safe C-to-Rust migration.

Translate the given C source file to idiomatic, safe Rust. Rules:
1. NEVER use `unsafe` — use Rust ownership/borrowing instead.
2. CBO (external crate imports) must be below 3.
3. Every public function must be semantically equivalent to its C counterpart.
4. Use idiomatic Rust: Option<T>, iterators, match expressions.
5. Output ONLY the complete .rs file. No markdown, no explanations."""

def build_prompt(obs):
    constraints = '\n'.join(f'  - {c}' for c in obs.get('constraints', []))
    dep_ctx = obs.get('dependency_context', {})
    dep_str = '\n'.join(f'// {f}:\n{s}' for f, s in dep_ctx.items()) or '// No prior translations yet.'
    errors  = obs.get('recent_errors', [])
    err_str = '\n'.join(f'  [{e.get("level")}] {e.get("message")}' for e in errors) or '  None'

    user_msg = (
        f"## Migration Task\nTranslate this C file to memory-safe Rust.\n\n"
        f"## Constraints\n{constraints}\n\n"
        f"## C Source: {obs.get('current_target')}\n```c\n{obs.get('c_source_code','')}\n```\n\n"
        f"## Already-Translated Context\n{dep_str}\n\n"
        f"## Recent Compiler Errors (fix these)\n{err_str}\n\n"
        f"Provide the complete Rust source file:"
    )
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': user_msg}],
        tokenize=False, add_generation_prompt=True
    )

def make_dataset(phase=1):
    env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
    prompts = []
    for c_set in [
        ['Do not use the unsafe keyword', 'Maintain a CBO score below 3'],
        ['Do not use the unsafe keyword'],
        ['Maintain a CBO score below 3'],
        ['Do not use the unsafe keyword', 'Maintain a CBO score below 3'],
    ]:
        obs = env.reset(constraints=c_set, phase=phase)
        if obs.get('current_target'):
            prompts.append(build_prompt(obs))
    return Dataset.from_dict({'prompt': prompts})

dataset = make_dataset(phase=1)
print(f'Dataset: {len(dataset)} prompts')
print(f'Sample prompt length: {len(dataset[0]["prompt"])} chars')
print('\n=== Sample prompt preview (first 600 chars) ===')
print(dataset[0]['prompt'][:600])

In [ ]:
# Reward function — fully programmatic (real Rust compiler, no LLM judge)
def reward_func(prompts=None, completions=None, **kwargs):
    rewards = []
    env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
    for completion in (completions or []):
        try:
            obs = env.reset(phase=1)
            target = obs.get('current_target', 'math_ops.c')
            rs_path = 'src/' + re.sub(r'\.c$', '.rs', os.path.basename(target))
            result  = env.step({'file_path': rs_path, 'code_content': completion})
            rewards.append(result['reward'])
        except Exception as e:
            print(f'[reward_func error] {e}')
            rewards.append(0.01)
    return rewards

# Sanity check
test_r = reward_func(completions=['pub fn add(a: i32, b: i32) -> i32 { a + b }'])
print(f'Sanity check reward: {test_r[0]}')
print('✅ Reward function ready!')

## Cell 6 — Measure BASELINE before training

In [ ]:
model.eval()

def generate(prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded    = tokenizer.decode(out[0], skip_special_tokens=True)
    input_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
sample_obs    = env.reset(phase=1)
sample_prompt = build_prompt(sample_obs)

print('===============================')
print('  BASELINE (zero-shot model)  ')
print('===============================')

baseline_rewards = []
baseline_stages  = []

for i in range(4):
    completion = generate(sample_prompt)
    obs2 = env.reset(phase=1)
    r    = env.step({'file_path': 'src/math_ops.rs', 'code_content': completion})
    baseline_rewards.append(r['reward'])
    baseline_stages.append(r['info']['verification']['stage'])
    print(f'  Run {i+1}: reward={r["reward"]:.4f}  stage={r["info"]["verification"]["stage"]}')

baseline_avg = sum(baseline_rewards) / len(baseline_rewards)
print(f'\n📊 Baseline average reward: {baseline_avg:.4f}')
print('\n⚠️  NOTE THIS NUMBER — needed for the before/after comparison!')

## Cell 7 — GRPO Training

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# T4  (16GB): batch=1, steps=100 → ~60 min
# A10G(24GB): batch=2, steps=200 → ~40 min
IS_A10G = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 20e9
BATCH   = 2 if IS_A10G else 1
STEPS   = 200 if IS_A10G else 100

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Using batch_size={BATCH}, max_steps={STEPS}')

training_args = GRPOConfig(
    output_dir                  = '/content/crust_outputs',
    learning_rate               = 2e-5,
    per_device_train_batch_size = BATCH,
    gradient_accumulation_steps = 4,
    max_steps                   = STEPS,
    logging_steps               = 5,
    save_steps                  = 50,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    num_generations             = 4,       # GRPO group size
    temperature                 = 0.8,
    max_completion_length       = 512,     # NOTE: NOT max_new_tokens (trl>=0.9 rename)
    report_to                   = 'none',
    seed                        = 42,
    bf16                        = IS_A10G,
    fp16                        = not IS_A10G,
)

model.train()

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = reward_func,
    args             = training_args,
    train_dataset    = dataset,
)

print(f'\n🚀 Starting GRPO training ({STEPS} steps)...')
print('Watch the reward column — it should trend upward!')
print('─' * 60)

trainer.train()

print('─' * 60)
print('✅ Training complete!')

## Cell 8 — Measure AFTER training + plot reward improvement

In [ ]:
import matplotlib.pyplot as plt

model.eval()

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])

print('===============================')
print('  TRAINED MODEL (post-GRPO)   ')
print('===============================')

trained_rewards = []
trained_stages  = []

for i in range(4):
    completion = generate(sample_prompt)
    obs2 = env.reset(phase=1)
    r    = env.step({'file_path': 'src/math_ops.rs', 'code_content': completion})
    trained_rewards.append(r['reward'])
    trained_stages.append(r['info']['verification']['stage'])
    print(f'  Run {i+1}: reward={r["reward"]:.4f}  stage={r["info"]["verification"]["stage"]}')

trained_avg = sum(trained_rewards) / len(trained_rewards)
delta = trained_avg - baseline_avg
pct   = (delta / max(baseline_avg, 0.01)) * 100

print(f'\n📊 Results:')
print(f'  Baseline avg : {baseline_avg:.4f}')
print(f'  Trained avg  : {trained_avg:.4f}')
print(f'  Improvement  : +{delta:.4f}  ({pct:+.1f}%)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('CRust: GRPO Training Results — Meta OpenEnv Hackathon', fontsize=13, fontweight='bold')

ax = axes[0]
bars = ax.bar(['Baseline\n(zero-shot)', 'CRust GRPO\n(trained)'],
              [baseline_avg, trained_avg],
              color=['#e74c3c', '#2ecc71'], width=0.5)
ax.set_ylabel('Average Reward (0–1)')
ax.set_title('Reward: Before vs After GRPO')
ax.set_ylim(0, 1.0)
ax.axhline(0.40, color='orange', linestyle='--', alpha=0.6, label='Compiles (0.40)')
ax.legend(fontsize=9)
for bar, val in zip(bars, [baseline_avg, trained_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', fontweight='bold', fontsize=13)

ax2 = axes[1]
all_s    = sorted(set(baseline_stages + trained_stages))
b_counts = {s: baseline_stages.count(s) for s in all_s}
t_counts = {s: trained_stages.count(s)  for s in all_s}
x = range(len(all_s))
ax2.bar([i - 0.2 for i in x], [b_counts.get(s, 0) for s in all_s], width=0.35,
        color='#e74c3c', label='Baseline', alpha=0.85)
ax2.bar([i + 0.2 for i in x], [t_counts.get(s, 0) for s in all_s], width=0.35,
        color='#2ecc71', label='Trained', alpha=0.85)
ax2.set_xticks(list(x)); ax2.set_xticklabels(all_s, rotation=15)
ax2.set_ylabel('Count'); ax2.set_title('Verification Stage Distribution')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/crust_reward_improvement.png', dpi=180, bbox_inches='tight')
plt.show()
print('\n💾 Chart saved to /content/crust_reward_improvement.png')

## Cell 9 — Side-by-side judge demo

In [ ]:
import json

print('=' * 70)
print('  JUDGE DEMO: Trained Model Generating a Rust Migration')
print('=' * 70)

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
obs = env.reset(phase=1, constraints=[
    'Do not use the unsafe keyword',
    'Maintain a CBO score below 3'
])

print(f'\nTask: Translate {obs["current_target"]} → Rust')
print(f'Constraints: {obs["constraints"]}')
print(f'\nC Source:\n{obs["c_source_code"]}\n')
print('─' * 70)

prompt    = build_prompt(obs)
rust_code = generate(prompt, max_new_tokens=600)
print('Generated Rust (trained model):')
print('─' * 70)
print(rust_code)

result = env.step({'file_path': 'src/math_ops.rs', 'code_content': rust_code})
print('\n─' * 70)
print(f'\n🎯 Reward: {result["reward"]}   Stage: {result["info"]["verification"]["stage"]}')
print(f'📐 Metrics: {result["info"]["metrics"]}')
print(f'💰 Breakdown:\n{json.dumps(result["info"]["reward_breakdown"], indent=2)}')

## Cell 10 — Save model and push to HF Hub

In [ ]:
from huggingface_hub import login

# Paste your token from https://huggingface.co/settings/tokens
HF_TOKEN    = 'YOUR_HF_TOKEN_HERE'
HF_USERNAME = 'Adithyakommuri'
REPO_NAME   = f'{HF_USERNAME}/crust-grpo-qwen25-3b'

login(token=HF_TOKEN, add_to_git_credential=False)

print('Saving LoRA adapters...')
model.save_pretrained('/content/crust_lora')
tokenizer.save_pretrained('/content/crust_lora')

print(f'Pushing to {REPO_NAME}...')
model.push_to_hub(REPO_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)

print(f'\n✅ Model saved to: https://huggingface.co/{REPO_NAME}')

## Cell 11 — Verify the live HF Space OpenEnv server

In [ ]:
import requests, json

BASE = 'https://adithyakommuri-meta-hackathon-final.hf.space'

print(f'Testing live OpenEnv server at {BASE}...')

h    = requests.get(f'{BASE}/health', timeout=15)
print('Health:', h.json())

info = requests.get(f'{BASE}/info', timeout=15)
print('\nEnvironment info:')
print(json.dumps(info.json(), indent=2)[:600])

obs = requests.post(f'{BASE}/reset', json={'phase': 1}, timeout=30).json()
print(f'\nReset OK — target: {obs.get("current_target")}')
print(f'Constraints: {obs.get("constraints")}')

final_result = requests.post(f'{BASE}/step', json={
    'file_path': 'src/math_ops.rs',
    'code_content': rust_code
}, timeout=60).json()

print(f'\n🎯 Live server reward: {final_result["reward"]}')
print(f'Stage: {final_result["info"]["verification"]["stage"]}')
print(f'\n✅ Live HF Space OpenEnv server is working!')
print(f'🔗 Share with judges: {BASE}/docs')